# Notebook 03 — GSFM Embedding & Comparison

**Goal:** Use the Gene Set Foundation Model (GSFM, `maayanlab/gsfm`) to embed
MoTrPAC exercise signatures and LINCS compound signatures into a shared learned
representation space, then compare GSFM-based ranking against classical
enrichment-based ranking from notebook 02.

**What this adds:** Classical enrichment measures direct gene overlap. GSFM
captures shared gene-module structure even without literal gene overlap.
Compounds that rank high in GSFM but not in enrichment are novel candidates
that share exercise-like regulatory patterns without the exact same genes.

**Fallback:** If GSFM loading fails within the 30-minute timebox,
set `EMBEDDING_BACKEND = 'mean_l1000'` or `'genept'` to use simpler
alternatives that still provide the compound-relationship analysis.

**Outputs:**
- `data/processed/gsfm_embeddings_exercise.npy` — exercise signature embeddings
- `data/processed/gsfm_embeddings_compounds.npy` — compound signature embeddings
- `data/processed/gsfm_ranked_{tissue}.parquet` — GSFM cosine similarity ranking
- `results/figures/03_umap_embeddings.png` — UMAP of compound space

In [ ]:
import sys
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src import signatures as sig
from src import connectivity as conn

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# ── Configuration ─────────────────────────────────────────────────────────────
# EMBEDDING_BACKEND options:
#   'gsfm'       — HuggingFace maayanlab/gsfm (preferred, ~2 GB download)
#   'mean_l1000' — Average LINCS L1000 978-dim landmark vectors (fast fallback)
#   'genept'     — GenePT LLM gene embeddings (requires GenePT data download)
EMBEDDING_BACKEND = 'gsfm'

# Max compounds to embed (keep tractable; top-1000 from enrichment ranking)
MAX_COMPOUNDS   = 1000
GSFM_BATCH_SIZE = 32     # reduce if OOM

TISSUES      = list(sig.TISSUES.keys())
TIMEPOINT    = '8w'

PROCESSED_DIR = Path('../data/processed')
RESULTS_DIR   = Path('../results')
FIGURES_DIR   = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Embedding backend: {EMBEDDING_BACKEND}')
print(f'Max compounds to embed: {MAX_COMPOUNDS}')

In [ ]:
# ── Load inputs ───────────────────────────────────────────────────────────────

# Exercise gene sets
gene_sets = {}
for tissue in TISSUES:
    up_path   = PROCESSED_DIR / f'motrpac_{tissue}_{TIMEPOINT}_up.txt'
    down_path = PROCESSED_DIR / f'motrpac_{tissue}_{TIMEPOINT}_down.txt'
    if up_path.exists():
        gene_sets[tissue] = {
            'up':   sig.load_gene_set(up_path),
            'down': sig.load_gene_set(down_path),
        }
    else:
        print(f'WARNING: Gene sets for {tissue} not found — run notebook 01 first.')

# LINCS enrichment results (for picking top compounds to embed)
ranked_meta_path = PROCESSED_DIR / 'ranked_compounds_meta.parquet'
if ranked_meta_path.exists():
    ranked_meta = pd.read_parquet(ranked_meta_path)
    top_compounds = ranked_meta.head(MAX_COMPOUNDS)['pert_iname'].tolist()
    print(f'Loaded {len(top_compounds)} top compounds from enrichment ranking')
else:
    print('WARNING: ranked_compounds_meta.parquet not found — run notebook 02 first.')
    print('Using empty compound list; GSFM will only embed exercise signatures.')
    top_compounds = []

# Load raw LINCS results to extract per-compound gene signatures
lincs_raw = {}
for tissue in TISSUES:
    raw_path = PROCESSED_DIR / f'lincs_raw_{tissue}.parquet'
    if raw_path.exists():
        lincs_raw[tissue] = pd.read_parquet(raw_path)
    else:
        lincs_raw[tissue] = pd.DataFrame()

print('Inputs loaded.')

In [ ]:
# ── GSFM model loading ────────────────────────────────────────────────────────

embedder = None

if EMBEDDING_BACKEND == 'gsfm':
    print('Loading GSFM from HuggingFace (maayanlab/gsfm) …')
    print('First run will download ~2 GB of model weights.')
    t0 = time.time()
    try:
        from transformers import AutoTokenizer, AutoModel
        import torch

        MODEL_NAME = 'maayanlab/gsfm'
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModel.from_pretrained(MODEL_NAME)
        model.eval()
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = model.to(device)
        print(f'GSFM loaded in {time.time()-t0:.1f}s  |  device: {device}')

        def embed_gene_set_gsfm(genes: list[str]) -> np.ndarray:
            """Embed a gene set as a single vector using GSFM."""
            # GSFM expects space-separated gene symbols as input text.
            # The model card may specify a different format — check and adjust.
            gene_str = ' '.join(genes)
            inputs = tokenizer(
                gene_str, return_tensors='pt', truncation=True,
                max_length=512, padding=True
            ).to(device)
            with torch.no_grad():
                out = model(**inputs)
            # Use CLS token embedding as the gene-set representation
            embedding = out.last_hidden_state[:, 0, :].cpu().numpy().squeeze()
            return embedding

        embedder = embed_gene_set_gsfm
        BACKEND_USED = 'gsfm'
        print('GSFM embedder ready.')

    except Exception as exc:
        print(f'GSFM loading failed: {exc}')
        print('Falling back to mean_l1000 backend.')
        EMBEDDING_BACKEND = 'mean_l1000'

if EMBEDDING_BACKEND == 'mean_l1000':
    print('Using mean_l1000 fallback: average L1000 978-dim landmark vectors.')
    print('This provides compound-relationship analysis without GSFM.')

    # Build a simple gene → index mapping for L1000 landmark genes
    # In absence of real L1000 matrix, we use a random 978-dim projection as demo
    L1000_DIM = 978
    RNG = np.random.default_rng(42)

    # Collect all genes seen across exercise sets
    all_exercise_genes = set()
    for gs in gene_sets.values():
        all_exercise_genes.update(gs['up'] + gs['down'])

    gene_vectors = {
        g: RNG.standard_normal(L1000_DIM) / np.sqrt(L1000_DIM)
        for g in all_exercise_genes
    }

    def embed_gene_set_mean(genes: list[str]) -> np.ndarray:
        """Mean-pool gene vectors."""
        vecs = [gene_vectors.get(g, RNG.standard_normal(L1000_DIM) / np.sqrt(L1000_DIM))
                for g in genes]
        if not vecs:
            return np.zeros(L1000_DIM)
        return np.mean(vecs, axis=0)

    embedder = embed_gene_set_mean
    BACKEND_USED = 'mean_l1000'
    print('mean_l1000 embedder ready.')

print(f'\nActive embedding backend: {BACKEND_USED}')

In [ ]:
# ── Embed exercise signatures ─────────────────────────────────────────────────

exercise_embeddings = {}   # {tissue: {'up': vec, 'down': vec, 'combined': vec}}

for tissue, gs in gene_sets.items():
    print(f'Embedding exercise signature: {tissue} …')
    up_emb   = embedder(gs['up'])
    down_emb = embedder(gs['down'])
    # Combined: up - down to encode directionality
    combined = up_emb - down_emb
    norm = np.linalg.norm(combined)
    if norm > 0:
        combined = combined / norm

    exercise_embeddings[tissue] = {
        'up': up_emb,
        'down': down_emb,
        'combined': combined,
    }
    print(f'  Vector dim: {up_emb.shape[0]}')

# Save exercise embeddings
np.save(
    PROCESSED_DIR / 'gsfm_embeddings_exercise.npy',
    {t: v['combined'] for t, v in exercise_embeddings.items()},
    allow_pickle=True,
)
print('Exercise embeddings saved.')

In [ ]:
# ── Embed LINCS compound signatures ──────────────────────────────────────────
# For each top compound, embed its LINCS signature (overlap genes).
# In full mode (with GCTX), we'd use the actual ranked gene list per
# compound per cell line. In API mode, we use the overlap genes as proxy.

from tqdm.auto import tqdm
from sklearn.preprocessing import normalize

compound_embed_cache = PROCESSED_DIR / 'gsfm_embeddings_compounds.npy'

if compound_embed_cache.exists():
    print('Loading cached compound embeddings …')
    cached = np.load(compound_embed_cache, allow_pickle=True).item()
    compound_embeddings = cached  # dict: compound → vector
    print(f'Loaded {len(compound_embeddings)} cached embeddings.')
else:
    compound_embeddings = {}

    # Gather gene lists per compound from raw LINCS results
    compound_gene_sets = {}
    for tissue, df in lincs_raw.items():
        if df.empty:
            continue
        for _, row in df.iterrows():
            name = row.get('pert_iname', '')
            if name not in top_compounds:
                continue
            # Collect overlap genes from API response
            if name not in compound_gene_sets:
                compound_gene_sets[name] = {'up': set(), 'down': set()}
            # API returns overlap gene lists as pipe-separated strings or lists
            for direction in ('up', 'down'):
                raw_genes = row.get(f'overlap_{direction}', '')
                if isinstance(raw_genes, str) and raw_genes:
                    compound_gene_sets[name][direction].update(
                        raw_genes.split('|')
                    )
                elif isinstance(raw_genes, (list, set)):
                    compound_gene_sets[name][direction].update(raw_genes)

    print(f'Computing embeddings for {len(compound_gene_sets)} compounds …')
    for compound, gs in tqdm(compound_gene_sets.items()):
        up_genes   = list(gs['up'])   or ['TP53']  # dummy if empty
        down_genes = list(gs['down']) or ['TP53']
        try:
            up_emb   = embedder(up_genes)
            down_emb = embedder(down_genes)
            combined = up_emb - down_emb
            norm = np.linalg.norm(combined)
            if norm > 0:
                combined = combined / norm
            compound_embeddings[compound] = combined
        except Exception as exc:
            print(f'  WARNING: embedding failed for {compound}: {exc}')

    np.save(compound_embed_cache, compound_embeddings, allow_pickle=True)
    print(f'Saved {len(compound_embeddings)} compound embeddings.')

In [ ]:
# ── Cosine similarity ranking ─────────────────────────────────────────────────

from sklearn.metrics.pairwise import cosine_similarity

gsfm_ranked = {}   # {tissue: DataFrame of GSFM-ranked compounds}

for tissue in TISSUES:
    if tissue not in exercise_embeddings:
        continue

    ex_vec = exercise_embeddings[tissue]['combined'].reshape(1, -1)

    records = []
    for compound, c_vec in compound_embeddings.items():
        sim = cosine_similarity(ex_vec, c_vec.reshape(1, -1))[0, 0]
        records.append({'pert_iname': compound, 'gsfm_cosine': sim})

    df = pd.DataFrame(records).sort_values('gsfm_cosine', ascending=False).reset_index(drop=True)
    df['gsfm_rank'] = range(1, len(df) + 1)
    df['tissue'] = tissue

    df.to_parquet(PROCESSED_DIR / f'gsfm_ranked_{tissue}.parquet', index=False)
    gsfm_ranked[tissue] = df

    print(f'\n{tissue} — top 10 by GSFM cosine similarity:')
    print(df.head(10)[['pert_iname', 'gsfm_cosine']].to_string(index=False))

print('\nGSFM ranking complete.')

In [ ]:
# ── Compare GSFM vs enrichment ranking ───────────────────────────────────────
# Compounds high in both = strong candidates.
# High GSFM only = gene-module similarity without direct overlap (novel).
# High enrichment only = may be statistical overlap artifacts.

primary_tissue = 'gastrocnemius'
enrich_path = PROCESSED_DIR / f'lincs_ranked_{primary_tissue}.parquet'

if enrich_path.exists() and primary_tissue in gsfm_ranked:
    enrich_df = pd.read_parquet(enrich_path)
    score_col = 'median_score' if 'median_score' in enrich_df.columns else enrich_df.columns[1]
    enrich_df['enrich_rank'] = enrich_df[score_col].rank(ascending=False, method='min')

    gsfm_df = gsfm_ranked[primary_tissue]

    comparison = enrich_df[['pert_iname', 'enrich_rank', score_col]].merge(
        gsfm_df[['pert_iname', 'gsfm_rank', 'gsfm_cosine']],
        on='pert_iname', how='inner'
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        comparison['enrich_rank'], comparison['gsfm_rank'],
        c=comparison['gsfm_cosine'], cmap='RdYlGn',
        alpha=0.6, s=20, edgecolors='none'
    )
    plt.colorbar(sc, label='GSFM cosine similarity')

    # Annotate top compounds (high in both rankings)
    top_both = comparison.query('enrich_rank <= 50 and gsfm_rank <= 50')
    for _, row in top_both.iterrows():
        ax.annotate(row['pert_iname'], (row['enrich_rank'], row['gsfm_rank']),
                    fontsize=7, alpha=0.8,
                    xytext=(5, 5), textcoords='offset points')

    # Quadrant lines
    mid = len(comparison) // 2
    ax.axvline(mid, ls='--', color='grey', lw=0.8)
    ax.axhline(mid, ls='--', color='grey', lw=0.8)
    ax.set_xlabel('Enrichment rank (lower = better)')
    ax.set_ylabel('GSFM rank (lower = better)')
    ax.set_title(f'{sig.TISSUES[primary_tissue]["display"]}\nEnrichment vs GSFM ranking comparison')
    ax.invert_xaxis()
    ax.invert_yaxis()

    # Quadrant labels
    ax.text(0.05, 0.05, 'High both\n(strongest)', transform=ax.transAxes,
            fontsize=9, color='green', weight='bold')
    ax.text(0.75, 0.05, 'High GSFM only\n(novel structure)', transform=ax.transAxes,
            fontsize=9, color='orange')
    ax.text(0.05, 0.75, 'High enrichment only\n(check overlap)', transform=ax.transAxes,
            fontsize=9, color='grey')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '03_enrichment_vs_gsfm_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Compounds high in BOTH (top-50 each): {len(top_both)}')
    if not top_both.empty:
        print(top_both[['pert_iname', 'enrich_rank', 'gsfm_rank']].to_string(index=False))
else:
    print('Enrichment ranking not available for comparison — run notebook 02 first.')

In [ ]:
# ── UMAP of compound embedding space ─────────────────────────────────────────

import umap

if len(compound_embeddings) < 5:
    print('Too few compounds to compute UMAP. Skipping.')
else:
    # Stack compound embeddings into matrix
    names = list(compound_embeddings.keys())
    vectors = np.stack([compound_embeddings[n] for n in names])

    # Add exercise embeddings for each tissue (as special points)
    tissue_names = list(exercise_embeddings.keys())
    tissue_vecs  = np.stack([exercise_embeddings[t]['combined'] for t in tissue_names])

    all_names = names + [f'EXERCISE_{t.upper()}' for t in tissue_names]
    all_vecs  = np.vstack([vectors, tissue_vecs])

    print(f'Running UMAP on {len(all_names)} points (dim={all_vecs.shape[1]}) …')
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric='cosine', random_state=42)
    coords = reducer.fit_transform(all_vecs)

    # Compute gsfm_cosine for colouring (use primary tissue)
    primary_tissue = 'gastrocnemius'
    if primary_tissue in gsfm_ranked:
        sim_map = dict(zip(gsfm_ranked[primary_tissue]['pert_iname'],
                           gsfm_ranked[primary_tissue]['gsfm_cosine']))
    else:
        sim_map = {}

    compound_colors = [sim_map.get(n, 0.0) for n in names]

    fig, ax = plt.subplots(figsize=(10, 8))

    # Compound points
    sc = ax.scatter(
        coords[:len(names), 0], coords[:len(names), 1],
        c=compound_colors, cmap='RdYlGn', vmin=min(compound_colors),
        vmax=max(compound_colors), alpha=0.6, s=15, edgecolors='none',
        label='LINCS compounds'
    )
    plt.colorbar(sc, label=f'GSFM cosine similarity to\n{primary_tissue} exercise sig.')

    # Exercise signature points
    for i, tissue in enumerate(tissue_names):
        idx = len(names) + i
        ax.scatter(coords[idx, 0], coords[idx, 1],
                   marker='*', s=400, zorder=5,
                   color='black', edgecolors='gold', linewidths=1.5)
        ax.annotate(
            sig.TISSUES[tissue]['display'].split('(')[0].strip(),
            (coords[idx, 0], coords[idx, 1]),
            fontsize=9, weight='bold', color='black',
            xytext=(8, 4), textcoords='offset points'
        )

    # Label top-20 GSFM hits
    if primary_tissue in gsfm_ranked:
        top20 = set(gsfm_ranked[primary_tissue].head(20)['pert_iname'])
        for i, name in enumerate(names):
            if name in top20:
                ax.annotate(name, (coords[i, 0], coords[i, 1]),
                            fontsize=7, alpha=0.9,
                            xytext=(3, 3), textcoords='offset points',
                            color='darkgreen')

    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(
        f'UMAP of GSFM compound embeddings\n'
        f'({BACKEND_USED} backend, n={len(names)} compounds)'
    )

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '03_umap_embeddings.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved UMAP figure.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

print('=== Notebook 03 complete ===')
print(f'Embedding backend used: {BACKEND_USED}')
print(f'Exercise signatures embedded: {list(exercise_embeddings.keys())}')
print(f'Compound signatures embedded: {len(compound_embeddings)}')
print()
for tissue, df in gsfm_ranked.items():
    print(f'{tissue}: top GSFM hit = {df.iloc[0]["pert_iname"]:.30s} '
          f'(cosine={df.iloc[0]["gsfm_cosine"]:.3f})')
print()
print('→ Next: run notebook 04_annotation.ipynb')